In [ ]:
import numpy as np
import plotly.graph_objects as go

# Interactive 3D visualization of nested generalized eigenspaces.
# Mathematical note:
# True generalized eigenspaces are linear subspaces:
#   G_k(λ) = null((T - λI)^k).
# This visualization uses translucent concentric spheres as a geometric metaphor for the subspaces
# for the nesting, while also drawing the actual nested subspaces:
#   G1 = a line
#   G2 = a plane
#   G3 = all of R^3 (shown by the outer sphere / ambient space cue)

# -------------------------
# Helper functions
# -------------------------
def sphere(radius, nu=50, nv=50):
    u = np.linspace(0, 2 * np.pi, nu)
    v = np.linspace(0, np.pi, nv)
    x = radius * np.outer(np.cos(u), np.sin(v))
    y = radius * np.outer(np.sin(u), np.sin(v))
    z = radius * np.outer(np.ones_like(u), np.cos(v))
    return x, y, z


def plane_patch(size=2.0, n=25):
    s = np.linspace(-size, size, n)
    X, Y = np.meshgrid(s, s)
    Z = np.zeros_like(X)
    return X, Y, Z


# -------------------------
# Build figure
# -------------------------
fig = go.Figure()

# Ambient concentric shells (metaphor for nesting)
radii = [0.8, 1.6, 2.4]
shell_names = [
    "G₁(λ): eigenspace / order 1",
    "G₂(λ): generalized eigenspace / order 2",
    "G₃(λ): generalized eigenspace / order 3",
]
# Using Plotly default palette manually avoided; choose readable RGBA values
shell_colors = [
    "rgba(31,119,180,0.18)",
    "rgba(44,160,44,0.14)",
    "rgba(255,127,14,0.10)",
]

for r, name, color in zip(radii, shell_names, shell_colors):
    x, y, z = sphere(r)
    fig.add_trace(
        go.Surface(
            x=x,
            y=y,
            z=z,
            opacity=float(color.split(',')[-1].replace(')', '')),
            showscale=False,
            showlegend=True,
            name=name,
            hoverinfo="skip",
            surfacecolor=np.zeros_like(x),
            colorscale=[[0, color], [1, color]],
            contours={
                "x": {"show": False},
                "y": {"show": False},
                "z": {"show": False},
            },
        )
    )

# Actual nested subspaces in R^3
# G1: line along z-axis
z_line = np.linspace(-2.6, 2.6, 200)
fig.add_trace(
    go.Scatter3d(
        x=np.zeros_like(z_line),
        y=np.zeros_like(z_line),
        z=z_line,
        mode="lines",
        line=dict(width=10, color="crimson"),
        name="Actual G₁: line",
        hovertemplate="Actual G₁ = line<extra></extra>",
    )
)

# G2: xy-plane
X, Y, Z = plane_patch(size=2.0)
fig.add_trace(
    go.Surface(
        x=X,
        y=Y,
        z=Z,
        opacity=0.28,
        showscale=False,
        name="Actual G₂: plane",
        hoverinfo="skip",
        surfacecolor=np.zeros_like(X),
        colorscale=[[0, "rgba(148,103,189,0.28)"], [1, "rgba(148,103,189,0.28)"]],
    )
)

# Origin marker
fig.add_trace(
    go.Scatter3d(
        x=[0], y=[0], z=[0],
        mode="markers+text",
        marker=dict(size=6, color="gold"),
        text=["origin"],
        textposition="top center",
        name="Origin",
        hovertemplate="Origin<extra></extra>",
    )
)

# Axis lines
axis_len = 2.8
fig.add_trace(go.Scatter3d(x=[-axis_len, axis_len], y=[0, 0], z=[0, 0], mode="lines", line=dict(width=4, color="black"), name="x-axis", hoverinfo="skip", showlegend=False))
fig.add_trace(go.Scatter3d(x=[0, 0], y=[-axis_len, axis_len], z=[0, 0], mode="lines", line=dict(width=4, color="black"), name="y-axis", hoverinfo="skip", showlegend=False))
fig.add_trace(go.Scatter3d(x=[0, 0], y=[0, 0], z=[-axis_len, axis_len], mode="lines", line=dict(width=4, color="black"), name="z-axis", hoverinfo="skip", showlegend=False))

# Labels placed in space
fig.add_trace(
    go.Scatter3d(
        x=[0.95, 1.75, 2.5],
        y=[0.0, 0.0, 0.0],
        z=[0.95, 1.7, 2.45],
        mode="text",
        text=["G₁", "G₂", "G₃"],
        textfont=dict(size=16, color="black"),
        showlegend=False,
        hoverinfo="skip",
    )
)

fig.update_layout(
    title=(
        "Nested Generalized Eigenspaces in 3D<br>"
        "Spheres show concentric nesting metaphor; line and plane show actual subspaces"
    ),
    width=950,
    height=800,
    scene=dict(
        xaxis=dict(title="x", range=[-3, 3], backgroundcolor="rgb(245,245,245)"),
        yaxis=dict(title="y", range=[-3, 3], backgroundcolor="rgb(245,245,245)"),
        zaxis=dict(title="z", range=[-3, 3], backgroundcolor="rgb(245,245,245)"),
        aspectmode="cube",
        camera=dict(eye=dict(x=1.5, y=1.4, z=1.1)),
    ),
    legend=dict(x=0.02, y=0.98),
    margin=dict(l=0, r=0, t=70, b=0),
)

fig.show()

# Optional: save as an interactive HTML file
# fig.write_html("generalized_eigenspaces_3d.html")
